In [12]:
import json
import spacy
import  os
import re  

In [13]:


def extract_reason_with_next_sentence_spacy(paragraph):
    """
    Use spaCy for better sentence detection
    """
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(paragraph)
    
    # Get all sentences
    sentences = list(doc.sents)

    # print("Sentences detected:")
    # for i, sent in enumerate(sentences):
    #     print(f"  {i}: {sent.text.strip()}")    
    
    trigger_phrases = [
        "because",
        "because of",
        "due to",
        "caused by",
        "resulted from",
    ]
    
    for i, sent in enumerate(sentences):
        sent_text = sent.text.strip()
        
        # Check if trigger phrase exists
        for trigger in trigger_phrases:
            if re.search(trigger, sent_text, re.IGNORECASE):
                matched_sentence = sent_text
                
                # Get next sentence
                next_sentence = sentences[i + 1].text.strip() if i + 1 < len(sentences) else ""
                
                combined = matched_sentence
                if next_sentence:
                    combined += " " + next_sentence
                
                return {
                    "matched_sentence": matched_sentence,
                    "next_sentence": next_sentence,
                    "combined": combined,
                    "trigger_found": trigger
                }
    
    return None





In [ ]:
def load_keyword_dictionary(filepath):
    """Load the keyword dictionary from JSON file"""
    with open(filepath, 'r') as f:
        keyword_dict = json.load(f)
    return keyword_dict

def setup_custom_ner_with_tokens(keyword_dict_path):
    nlp = spacy.load("en_core_web_sm")
    ruler = nlp.add_pipe("entity_ruler", before="ner")
    
    # Load the dictionary
    keyword_dict = load_keyword_dictionary(keyword_dict_path)
    
    patterns = []
    
    # Method: Token-based patterns from values (is more flexible)
    for reason_phrase, tokens in keyword_dict.items():
        token_pattern = [{"LOWER": token.lower()} for token in tokens]
        patterns.append({
            "label": "ADVISORY_REASON",
            "pattern": token_pattern
        })
    
    ruler.add_patterns(patterns)


    print(f"Loaded {len(patterns)} patterns into the EntityRuler.")
    print("pattern : ", patterns   )


    return nlp

In [ ]:
import os
import json

INPUT_FOLDER = "processed_json"
OUTPUT_FOLDER = "advisory_reasons_output"

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def process_all_json_files_sequential(input_folder, output_folder):
    """
    Process files one by one 
    """
    # Load NLP model ONCE
    print("Loading NLP model...")
    nlp = setup_custom_ner_with_tokens("reason_dictionary.json")
    
    # Get all JSON files
    json_files = [
        f for f in os.listdir(input_folder)
        if f.endswith(".json")
    ]
    
    total_files = len(json_files)
    print(f"Found {total_files} JSON files")
    print("="*50)
    
    all_file_results = []
    
    for file_idx, file_name in enumerate(json_files, 1):
        print(f"[{file_idx}/{total_files}] Processing: {file_name}")
        
        file_path = os.path.join(input_folder, file_name)
        
        # Load JSON data
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if isinstance(data, list):
            advisories = data
        else:
            advisories = [data]
        
        file_results = []
        
        # Process each advisory in this file
        for i, advisory in enumerate(advisories):
            try:
                # Extract context
                paraExtract = extract_reason_with_next_sentence_spacy(advisory["paragraph"])
                combinedContext = paraExtract['combined'] if paraExtract else advisory["paragraph"]
                
                # Process with NLP
                doc = nlp(combinedContext)
                
                advisory_reasons = []
                for ent in doc.ents:
                    if ent.label_ == "ADVISORY_REASON":
                        advisory_reasons.append(ent.text)
                
                result = {
                    "url": advisory.get("url"),
                    "title": advisory.get("title"),
                    "combined_context": combinedContext,
                    "extracted_entities_advisory_reason": advisory_reasons,
                    "posted_on": advisory.get("posted_on"),
                    "issued_date": advisory.get("issued_date"),
                    "rescinded_date": advisory.get("rescinded_date"),
                    "city": advisory.get("city"),
                    "county": advisory.get("county")
                }
                
                file_results.append(result)
                
            except Exception as e:
                print(f"  Error in advisory {i+1}: {e}")
                continue
        
        # Save output for this file
        output_path = os.path.join(output_folder, f"extracted_{file_name}")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(file_results, f, indent=2, ensure_ascii=False)
        
        print(f" Saved {len(file_results)} advisories to {output_path}")
        
        all_file_results.append({
            "file": file_name,
            "count": len(file_results)
        })
    
    # Print summary
    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Files processed: {len(all_file_results)}")
    total_advisories = sum(r['count'] for r in all_file_results)
    print(f"Total advisories: {total_advisories}")
    print(f"Output folder: {output_folder}")
    
    return all_file_results


# Usage:
if __name__ == "__main__":
    results = process_all_json_files_sequential(INPUT_FOLDER, OUTPUT_FOLDER)

Loading NLP model...
Loaded 56 patterns into the EntityRuler.
pattern :  [{'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'waterline'}, {'LOWER': 'break'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'loss'}, {'LOWER': 'of'}, {'LOWER': 'pressure'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'low'}, {'LOWER': 'cl'}, {'LOWER': 'residual'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'turbidity'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'manganese'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'nitrate'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'damage'}, {'LOWER': 'to'}, {'LOWER': 'water'}, {'LOWER': 'main'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'low'}, {'LOWER': 'cl'}, {'LOWER': 'residual'}, {'LOWER': 'do'}, {'LOWER': 'not'}, {'LOWER': 'drink'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'high'}, {'LOWER': 'turbidity'}]}, {'label': 'ADVISORY_REASON', 'pattern': [{'LOWER': 'contaminated'}, {'LOWER